In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [ ]:
import json, sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "corpus").exists() else Path.cwd().parent
sys.path.append(str(ROOT / "scripts"))
from chunking import chunk_words
from golden import facts, load_golden, norm, recall_at_k

docs = {p.stem: p.read_text(encoding="utf-8") for p in sorted((ROOT / "corpus/text").glob("*.txt"))}
golden = load_golden(ROOT / "eval/golden.jsonl")
print(len(docs), "documents,", len(golden), "questions")

In [ ]:
import pandas as pd

fact_list = [(g["id"], alts) for g in golden for alts in facts(g)]
CONFIGS = [(64, 0), (64, 32), (128, 0), (128, 32), (192, 0), (192, 48)]

def build_chunks(size, overlap):
    # chunk each document separately so no chunk mixes two documents
    return [{"doc": name, "text": c} for name, t in docs.items() for c in chunk_words(t, size, overlap)]

rows = []
for size, ov in CONFIGS:
    texts = [norm(c["text"]) for c in build_chunks(size, ov)]
    covered = sum(any(norm(a) in t for a in alts for t in texts) for _, alts in fact_list)
    rows.append({"size": size, "overlap": ov, "chunks": len(texts), "facts_covered": f"{covered}/{len(fact_list)}"})
pd.DataFrame(rows)

In [ ]:
from transformers import AutoTokenizer

MODELS = {
    "e5": ("intfloat/multilingual-e5-base", "passage: "),
    "tr": ("trmteb/turkish-embedding-model", ""),
}
toks = {k: AutoTokenizer.from_pretrained(name) for k, (name, _) in MODELS.items()}

rows = []
for size, ov in CONFIGS:
    chunks = build_chunks(size, ov)
    row = {"size": size, "overlap": ov}
    for k, (_, prefix) in MODELS.items():
        lens = [len(toks[k](prefix + c["text"])["input_ids"]) for c in chunks]
        row[f"{k}_max"] = max(lens)
        row[f"{k}_over_512"] = sum(l > 512 for l in lens)
    rows.append(row)
pd.DataFrame(rows)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

RUN_CONFIGS = [(64, 0), (64, 32), (128, 32), (192, 48)]
QUERY_PREFIX = {"e5": "query: ", "tr": ""}
models = {k: SentenceTransformer(name) for k, (name, _) in MODELS.items()}

def embed(k, texts, is_query):
    prefix = QUERY_PREFIX[k] if is_query else MODELS[k][1]
    return models[k].encode([prefix + t for t in texts], normalize_embeddings=True,
                            batch_size=32, show_progress_bar=False)

# rankings[(model, size, overlap)][question_id] = chunk texts sorted from most to least similar
rankings = {}
for size, ov in RUN_CONFIGS:
    chunks = build_chunks(size, ov)
    texts = [c["text"] for c in chunks]
    for k in MODELS:
        chunk_vecs = embed(k, texts, is_query=False)
        q_vecs = embed(k, [g["question"] for g in golden], is_query=True)
        scores = q_vecs @ chunk_vecs.T          # cosine similarity, vectors are normalized
        order = np.argsort(-scores, axis=1)[:, :10]
        rankings[(k, size, ov)] = {g["id"]: [texts[i] for i in order[n]] for n, g in enumerate(golden)}
        print(f"done: {k} size={size} overlap={ov} chunks={len(texts)}")

In [ ]:
rows = []
for (k, size, ov), ranks in rankings.items():
    row = {"model": k, "size": size, "overlap": ov}
    for K in (1, 3, 5, 10):
        scores = []
        for g in golden:
            scores.append(recall_at_k(ranks[g["id"]], facts(g), K))
        row[f"R@{K}"] = np.mean(scores)
    rows.append(row)
pd.DataFrame(rows).round(2)

In [ ]:
detail = {}
for (k, size, ov), ranks in rankings.items():
    col = f"{k}-{size}/{ov}"
    detail[col] = {
        g["id"]: recall_at_k(ranks[g["id"]], facts(g), 10)
        for g in golden
    }
d = pd.DataFrame(detail)
d.insert(0, "type", [g["type"] for g in golden])
d.insert(1, "question", [g["question"][:45] for g in golden])
d

In [ ]:
def show(qid, key=("e5", 192, 48), n=3):
    for i, t in enumerate(rankings[key][qid][:n], 1):
        print(i, t[:250], "\n")

show("q008")

In [ ]:
out = ROOT / "results"
out.mkdir(exist_ok=True)
pd.DataFrame(rows).round(2).to_csv(out / "v1_summary.csv", index=False)
d.to_csv(out / "v1_per_question.csv")
print("saved")